# client

> Thin wrapper around an OpenAI-compatible chat completions API. Knows nothing about notebooks or JupyterLab.

In [ ]:
#| default_exp client

In [ ]:
#| hide
from nbdev.showdoc import *

## Model listing

Different providers return different fields from `/models`: OpenRouter has `name` (`"Meta: Llama 3.1 8B"`), OpenAI has `owned_by`, Anthropic's OpenAI-compatible endpoint has `display_name`. `model_entry` tolerates all of them.

In [ ]:
#| export
def model_entry(model # A model object as returned by `client.models.list()`
               )->dict: # dict with keys company/name/id
    """Builds yasi's model dict from an API model object, tolerating provider-specific fields
    (OpenRouter: `name`, OpenAI: `owned_by`, Anthropic: `display_name`)"""
    name = getattr(model, 'name', None)
    display_name = getattr(model, 'display_name', None)
    owned_by = getattr(model, 'owned_by', None)
    if name: company = name.split(':')[0]
    elif owned_by: company = owned_by
    elif '/' in model.id: company = model.id.split('/')[0]
    else: company = model.id.split('-')[0]
    return {"company": company,
            "name": name if name else (display_name if display_name else model.id.split('/')[-1]),
            "id": model.id,
            "pricing": getattr(model, 'pricing', None)}

In [ ]:
from types import SimpleNamespace

# OpenRouter style (includes per-token pricing)
m = SimpleNamespace(id='meta-llama/llama-3.1-8b-instruct', name='Meta: Llama 3.1 8B Instruct',
                    pricing={'prompt': '0.00000002', 'completion': '0.00000003'})
e = model_entry(m)
assert (e['company'], e['name'], e['id']) == ('Meta', 'Meta: Llama 3.1 8B Instruct', 'meta-llama/llama-3.1-8b-instruct')
assert e['pricing'] == {'prompt': '0.00000002', 'completion': '0.00000003'}

# OpenAI style
m = SimpleNamespace(id='gpt-4o-mini', owned_by='openai', name=None)
assert model_entry(m) == {'company': 'openai', 'name': 'gpt-4o-mini', 'id': 'gpt-4o-mini', 'pricing': None}

# Anthropic (OpenAI-compatible endpoint) style
m = SimpleNamespace(id='claude-sonnet-5', display_name='Claude Sonnet 5')
assert model_entry(m) == {'company': 'claude', 'name': 'Claude Sonnet 5', 'id': 'claude-sonnet-5', 'pricing': None}

## Provider quirks

Anthropic's OpenAI-compatible endpoint only accepts `temperature` between 0 and 1 (yasi's slider goes to 2), requires `max_tokens`, and ignores `presence_penalty`. `prepare_params` adjusts the request parameters accordingly, based on the base url.

In [ ]:
#| export
ANTHROPIC_DEFAULT_MAX_TOKENS = 4096

def prepare_params(base_url, # base url of the api, used to detect the provider
                   model:str=None,
                   max_tokens:int=None,
                   temperature:float=None,
                   presence_penalty:float=None
                  )->dict: # keyword arguments for `chat.completions.create`
    """Adjusts request parameters to provider quirks (currently: Anthropic's temperature range,
    required max_tokens and unsupported presence_penalty)"""
    if 'anthropic' in str(base_url or ''):
        if temperature is not None: temperature = max(0.0, min(temperature, 1.0))
        if max_tokens is None: max_tokens = ANTHROPIC_DEFAULT_MAX_TOKENS
        presence_penalty = None
    return dict(model=model, max_tokens=max_tokens,
                temperature=temperature, presence_penalty=presence_penalty)

In [ ]:
# Anthropic: clamp temperature, default max_tokens, drop presence_penalty
p = prepare_params('https://api.anthropic.com/v1/', model='claude-sonnet-5',
                   temperature=1.7, presence_penalty=0.5)
assert p == {'model': 'claude-sonnet-5', 'max_tokens': 4096, 'temperature': 1.0, 'presence_penalty': None}

# other providers: untouched
p = prepare_params('https://openrouter.ai/api/v1', model='meta-llama/llama-3.1-8b-instruct',
                   temperature=1.7, presence_penalty=0.5)
assert p == {'model': 'meta-llama/llama-3.1-8b-instruct', 'max_tokens': None,
             'temperature': 1.7, 'presence_penalty': 0.5}

## Usage & cost tracking

Every successful chat call records its token usage in `ChatClient.usage_log`. `usage_totals` sums it up, and `estimate_cost` computes spend from per-token pricing when the provider exposes it in its models list (OpenRouter does; Anthropic and Ollama don't). `format_usage` renders a short status line for the yasi panel.

In [ ]:
#| export
def usage_totals(usage_log:list # List of usage dicts with prompt_tokens/completion_tokens
                )->dict: # Totals over the whole log
    """Sums token usage over a usage log"""
    pt = sum(u['prompt_tokens'] for u in usage_log)
    ct = sum(u['completion_tokens'] for u in usage_log)
    return {'prompt_tokens': pt, 'completion_tokens': ct, 'total_tokens': pt + ct}

def estimate_cost(usage_log:list, # List of usage dicts with model/prompt_tokens/completion_tokens
                  models:list # Model dicts as returned by `get_models` (with 'pricing')
                 ): # Estimated cost in USD, or None if no pricing information is available
    """Estimates spend from per-token pricing where available (e.g. OpenRouter)"""
    pricing_by_id = {m['id']: m.get('pricing') for m in (models or [])}
    cost, known = 0.0, False
    for u in usage_log:
        pricing = pricing_by_id.get(u.get('model'))
        if pricing and pricing.get('prompt') is not None:
            cost += u['prompt_tokens'] * float(pricing['prompt'])
            cost += u['completion_tokens'] * float(pricing.get('completion') or 0)
            known = True
    return cost if known else None

def format_usage(usage_log:list, # List of usage dicts
                 models:list=None # Model dicts, for cost estimation
                )->str: # One-line summary for display
    """Formats the last exchange and running totals (plus cost, if estimable) as one line"""
    if not usage_log: return ''
    last, totals = usage_log[-1], usage_totals(usage_log)
    text = (f"Last: {last['prompt_tokens']}\u2191 {last['completion_tokens']}\u2193 | "
            f"Total: {totals['prompt_tokens']}\u2191 {totals['completion_tokens']}\u2193 tokens")
    cost = estimate_cost(usage_log, models)
    if cost is not None:
        text += f" (~${cost:.4f})"
    return text

In [ ]:
log = [{'model': 'meta-llama/llama-3.1-8b-instruct', 'prompt_tokens': 100, 'completion_tokens': 50},
       {'model': 'meta-llama/llama-3.1-8b-instruct', 'prompt_tokens': 200, 'completion_tokens': 100}]
assert usage_totals(log) == {'prompt_tokens': 300, 'completion_tokens': 150, 'total_tokens': 450}

models = [{'id': 'meta-llama/llama-3.1-8b-instruct', 'pricing': {'prompt': '0.00000002', 'completion': '0.00000003'}}]
cost = estimate_cost(log, models)
assert abs(cost - (300*0.00000002 + 150*0.00000003)) < 1e-12
assert estimate_cost(log, [{'id': 'other'}]) is None
assert estimate_cost([], models) is None

line = format_usage(log, models)
assert line.startswith('Last: 200\u2191 100\u2193') and 'Total: 300\u2191 150\u2193' in line and '$' in line
assert '$' not in format_usage(log)  # no models, no cost
assert format_usage([]) == ''
line

In [ ]:
#| export
import openai

class ChatClient:
    """Wraps an `openai.Client` for listing models and sending chat completions."""
    def __init__(self,
                 api_key : str=None,  # api key for the openai api
                 openai_base_url : str=None # base url of the openai api
                ):
        self.client = openai.Client(base_url=openai_base_url, api_key=api_key)
        self.latest_response = None
        self.usage_log = []

    def get_models(self):
        """Create list of model objects from the Openai client"""
        return [model_entry(model) for model in self.client.models.list()]

    def _log_usage(self, model, usage):
        if usage is not None:
            self.usage_log.append({'model': model,
                                   'prompt_tokens': getattr(usage, 'prompt_tokens', 0) or 0,
                                   'completion_tokens': getattr(usage, 'completion_tokens', 0) or 0})

    def chat(self,
             messages: list, # A list of messages comprising the conversation so far.
             model: str=None, # model id for the openai api
             max_tokens: int=None,
             temperature: float=None,
             presence_penalty: float=None,
             stream: bool=False, # stream the response
             on_chunk=None # callback(text_so_far), called after every streamed chunk
            )->str: # The response text, or an error description
        """Send messages a.k.a dialog to the Openai chat completions API"""
        params = prepare_params(self.client.base_url,
                                model=model,
                                max_tokens=max_tokens,
                                temperature=temperature,
                                presence_penalty=presence_penalty)
        try:
            if stream:
                response_text, usage = '', None
                chunks = self.client.chat.completions.create(messages=messages, stream=True,
                                                             stream_options={'include_usage': True}, **params)
                for chunk in chunks:
                    self.latest_response = chunk
                    if getattr(chunk, 'usage', None) is not None:
                        usage = chunk.usage
                    if chunk.choices and getattr(chunk.choices[0].delta, 'content', None):
                        response_text += chunk.choices[0].delta.content
                        if on_chunk is not None:
                            on_chunk(response_text)
                self._log_usage(model, usage)
                response_text = response_text.strip()
            else:
                response = self.client.chat.completions.create(messages=messages, **params)
                self.latest_response = response
                self._log_usage(model, getattr(response, 'usage', None))
                response_text = response.choices[0].message.content.strip()
        except Exception as e:
            response_text = f'There was an problem communicating with the Openai API:\n\n{e}'
        return response_text

In [ ]:
from types import SimpleNamespace

def _chunk(text=None, usage=None):
    choices = [SimpleNamespace(delta=SimpleNamespace(content=text))] if text is not None else []
    return SimpleNamespace(choices=choices, usage=usage)

cc = ChatClient.__new__(ChatClient)
cc.usage_log, cc.latest_response = [], None
captured = {}
def fake_create(**kw):
    captured.update(kw)
    if kw.get('stream'):
        return iter([_chunk('Hel'), _chunk('lo'), _chunk(None, usage=SimpleNamespace(prompt_tokens=10, completion_tokens=5))])
    raise AssertionError('expected stream=True')
cc.client = SimpleNamespace(base_url='https://openrouter.ai/api/v1',
                            chat=SimpleNamespace(completions=SimpleNamespace(create=fake_create)))

seen = []
out = cc.chat([{'role': 'user', 'content': 'hi'}], model='m', stream=True, on_chunk=seen.append)
assert out == 'Hello'
assert seen == ['Hel', 'Hello']  # growing text after each chunk
assert cc.usage_log == [{'model': 'm', 'prompt_tokens': 10, 'completion_tokens': 5}]
assert captured['stream_options'] == {'include_usage': True}
print('streaming test passed')

Usage against OpenRouter (needs a network connection and an api key, so this cell is not executed during tests):

In [ ]:
#| notest
cc = ChatClient(openai_base_url="https://openrouter.ai/api/v1", api_key=None)
models = cc.get_models()
models[:3]

# Anthropic's OpenAI-compatible endpoint (note the trailing /v1/):
# cc = ChatClient(openai_base_url="https://api.anthropic.com/v1/", api_key=os.environ["ANTHROPIC_API_KEY"])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()